# Explorative Datenanalyse: S&P-500-Fundamentaldaten

Erster Blick auf `data/raw/fundamentals.csv`, bevor die Daten in `src/clean.py` bereinigt werden.

Leitfragen:
1. Wie viele Zeilen/Spalten hat die Tabelle überhaupt?
2. Wie heißen die Spalten genau?
3. Gibt es fehlende Werte?
4. Sehen die Zahlen plausibel aus?

In [1]:
from pathlib import Path

import pandas as pd

# Projektwurzel finden (der Ordner mit pyproject.toml), damit das Notebook
# unabhängig vom aktuellen Arbeitsverzeichnis funktioniert
ROOT = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / "pyproject.toml").exists())
RAW = ROOT / "data" / "raw" / "fundamentals.csv"

# Anzeigeoptionen: mehr Zeilen zeigen und große Zahlen ohne wissenschaftliche Notation
pd.set_option("display.max_rows", 100)
pd.set_option("display.float_format", "{:,.2f}".format)

df = pd.read_csv(RAW)

## 1. Wie viele Zeilen/Spalten hat die Tabelle?

In [2]:
# shape liefert (Anzahl Zeilen, Anzahl Spalten)
n_rows, n_cols = df.shape
print(f"Zeilen:  {n_rows:,}")
print(f"Spalten: {n_cols}")

# Eine Zeile entspricht einem Jahresabschluss eines Unternehmens.
# Daher interessiert auch: wie viele Unternehmen und wie viele Abschlüsse je Unternehmen?
# Ticker = Börsenkürzel, unter dem eine Aktie gehandelt wird
print(f"Unternehmen (eindeutige Ticker): {df['Ticker Symbol'].nunique()}")
print("\nAbschlüsse je Unternehmen -> Anzahl Unternehmen:")
print(df["Ticker Symbol"].value_counts().value_counts().sort_index().to_string())

Zeilen:  1,781
Spalten: 79
Unternehmen (eindeutige Ticker): 448

Abschlüsse je Unternehmen -> Anzahl Unternehmen:
count
2      3
3      5
4    440


In [3]:
# Die ersten Zeilen geben ein Gefühl für den Aufbau der Daten
df.head()

,Unnamed: 0,Ticker Symbol,Period Ending,Accounts Payable,Accounts Receivable,Add'l income/expense items,After Tax ROE,Capital Expenditures,Capital Surplus,Cash Ratio,...,Total Current Assets,Total Current Liabilities,Total Equity,Total Liabilities,Total Liabilities & Equity,Total Revenue,Treasury Stock,For Year,Earnings Per Share,Estimated Shares Outstanding
0,0,AAL,2012-12-31,"3,068,000,000.00","-222,000,000.00","-1,961,000,000.00",23.00,"-1,888,000,000.00","4,695,000,000.00",53.00,...,"7,072,000,000.00","9,011,000,000.00","-7,987,000,000.00","24,891,000,000.00","16,904,000,000.00","24,855,000,000.00","-367,000,000.00","2,012.00",-5.60,"335,000,000.00"
1,1,AAL,2013-12-31,"4,975,000,000.00","-93,000,000.00","-2,723,000,000.00",67.00,"-3,114,000,000.00","10,592,000,000.00",75.00,...,"14,323,000,000.00","13,806,000,000.00","-2,731,000,000.00","45,009,000,000.00","42,278,000,000.00","26,743,000,000.00",0.00,"2,013.00",-11.25,"163,022,222.22"
2,2,AAL,2014-12-31,"4,668,000,000.00","-160,000,000.00","-150,000,000.00",143.00,"-5,311,000,000.00","15,135,000,000.00",60.00,...,"11,750,000,000.00","13,404,000,000.00","2,021,000,000.00","41,204,000,000.00","43,225,000,000.00","42,650,000,000.00",0.00,"2,014.00",4.02,"716,915,422.89"
3,3,AAL,2015-12-31,"5,102,000,000.00","352,000,000.00","-708,000,000.00",135.00,"-6,151,000,000.00","11,591,000,000.00",51.00,...,"9,985,000,000.00","13,605,000,000.00","5,635,000,000.00","42,780,000,000.00","48,415,000,000.00","40,990,000,000.00",0.00,"2,015.00",11.39,"668,129,938.54"
4,4,AAP,2012-12-29,"2,409,453,000.00","-89,482,000.00","600,000.00",32.00,"-271,182,000.00","520,215,000.00",23.00,...,"3,184,200,000.00","2,559,638,000.00","1,210,694,000.00","3,403,120,000.00","4,613,814,000.00","6,205,003,000.00","-27,095,000.00","2,012.00",5.29,"73,283,553.88"


## 2. Wie heißen die Spalten genau?

In [4]:
# Übersicht aller Spalten mit Datentyp und einem Beispielwert aus der ersten Zeile.
# Die Namen enthalten Leerzeichen und Sonderzeichen (z.B. "Sales, General and Admin."),
# deshalb werden sie in clean.py später in snake_case umbenannt.
columns = pd.DataFrame({
    "spalte": df.columns,
    "datentyp": df.dtypes.astype(str).values,
    "beispielwert": df.iloc[0].values,
})
columns

,spalte,datentyp,beispielwert
0,Unnamed: 0,int64,0
1,Ticker Symbol,str,AAL
2,Period Ending,str,2012-12-31
3,Accounts Payable,float64,"3,068,000,000.00"
4,Accounts Receivable,float64,"-222,000,000.00"
5,Add'l income/expense items,float64,"-1,961,000,000.00"
6,After Tax ROE,float64,23.00
7,Capital Expenditures,float64,"-1,888,000,000.00"
8,Capital Surplus,float64,"4,695,000,000.00"
9,Cash Ratio,float64,53.00


In [5]:
# "Unnamed: 0" ist nur der mitexportierte Index der Originaldatei (0, 1, 2, ...) und kein echtes Merkmal
print("Unnamed: 0 entspricht der Zeilennummer:", (df["Unnamed: 0"] == range(len(df))).all())

# "Period Ending" wurde als Text eingelesen -> muss später in ein Datum umgewandelt werden
print("Datentyp von Period Ending:", df["Period Ending"].dtype)

Unnamed: 0 entspricht der Zeilennummer: True
Datentyp von Period Ending: str


## 3. Gibt es fehlende Werte?

In [6]:
# Fehlende Werte je Spalte, absolut und in Prozent; nur Spalten mit Lücken anzeigen
missing = pd.DataFrame({
    "fehlend": df.isna().sum(),
    "fehlend_pct": df.isna().mean() * 100,
})
missing[missing["fehlend"] > 0].sort_values("fehlend", ascending=False)

,fehlend,fehlend_pct
Cash Ratio,299,16.79
Current Ratio,299,16.79
Quick Ratio,299,16.79
Earnings Per Share,219,12.30
Estimated Shares Outstanding,219,12.30
For Year,173,9.71


In [7]:
# Bei welchen Unternehmen fehlt die Current Ratio? Vermutung: Banken/Versicherer,
# die keine Trennung in kurz- und langfristige Bilanzposten ausweisen.
df.loc[df["Current Ratio"].isna(), "Ticker Symbol"].unique()

<StringArray>
[ 'AFL',  'AIG',  'AIV',  'AIZ',  'AJG',  'ALL',  'AMG',  'AMP', 'ANTM',
  'AON',  'AVB',  'AXP',  'BAC',  'BBT',   'BK',  'BXP',    'C',   'CB',
  'CFG',   'CI', 'CINF',  'CMA',  'CME',  'CNC',  'COF',  'DFS',  'DHI',
  'DLR',  'EQR',  'ESS', 'ETFC',  'EXR',  'FRT',  'GGP', 'HBAN',  'HCN',
  'HCP',  'HIG',  'HUM',  'JPM',  'KEY',  'KIM',  'LEN',  'MAA',  'MAC',
  'MET',  'MMC',  'MTB', 'NAVI', 'NDAQ', 'NTRS',    'O', 'PBCT',  'PFG',
  'PGR',  'PHM',  'PNC',  'PRU', 'SCHW',  'SLG',  'SPG',  'STI',  'STT',
  'SYF',  'TMK',  'TRV',  'UDR',  'UNH',  'UNM',  'VNO',  'VTR',  'WFC',
 'WLTW',   'XL', 'ZION']
Length: 75, dtype: str

In [8]:
# Neben echten Lücken (NaN) gibt es versteckte Probleme wie Duplikate.
# Exakte Duplikate (komplett identische Zeilen):
print("Exakte Duplikate:", df.duplicated().sum())

# Mehrere Abschlüsse für dasselbe Unternehmen im selben "For Year"
# -> Hinweis darauf, dass die Spalte "For Year" nicht zuverlässig ist
dup_year = df[df["For Year"].notna() & df.duplicated(["Ticker Symbol", "For Year"], keep=False)]
print("Zeilen mit doppelter Ticker/For-Year-Kombination:", len(dup_year))
dup_year[["Ticker Symbol", "Period Ending", "For Year", "Total Revenue"]].head(6)

Exakte Duplikate: 0
Zeilen mit doppelter Ticker/For-Year-Kombination: 12


,Ticker Symbol,Period Ending,For Year,Total Revenue
129,ANTM,2013-12-31,"2,013.00","71,023,500,000.00"
130,ANTM,2014-12-31,"2,013.00","73,874,100,000.00"
536,EIX,2012-12-31,"2,012.00","11,862,000,000.00"
537,EIX,2013-12-31,"2,012.00","12,581,000,000.00"
923,KORS,2013-03-30,"2,013.00","2,181,732,000.00"
924,KORS,2014-03-29,"2,013.00","3,310,800,000.00"


## 4. Sehen die Zahlen plausibel aus?

In [9]:
# Lageparameter der Kernspalten in Milliarden USD, damit die Größenordnung lesbar ist
key_cols = ["Total Revenue", "Cost of Revenue", "Gross Profit", "Net Income", "Total Assets"]
(df[key_cols] / 1e9).describe().T

,count,mean,std,min,25%,50%,75%,max
Total Revenue,"1,781.00",20.29,40.95,0.00,3.71,8.02,17.49,485.65
Cost of Revenue,"1,781.00",12.35,30.41,0.00,1.19,3.69,9.80,365.09
Gross Profit,"1,781.00",7.19,13.61,-12.65,1.58,2.99,6.94,149.45
Net Income,"1,781.00",1.71,3.99,-23.53,0.35,0.69,1.70,53.39
Total Assets,"1,781.00",55.71,203.36,0.00,6.55,15.17,36.00,"2,572.27"


In [10]:
# Stichprobe mit bekannten Zahlen: Apples Umsatz lag im Geschäftsjahr 2015 bei rund 234 Mrd. USD
df.loc[df["Ticker Symbol"] == "AAPL", ["Period Ending", "Total Revenue", "Net Income"]]

,Period Ending,Total Revenue,Net Income
8,2013-09-28,"170,910,000,000.00","37,037,000,000.00"
9,2014-09-27,"182,795,000,000.00","39,510,000,000.00"
10,2015-09-26,"233,715,000,000.00","53,394,000,000.00"
11,2016-09-24,"215,639,000,000.00","45,687,000,000.00"


In [11]:
# Plausibilitätsregeln: jede Prüfung zählt die Zeilen, die gegen eine Erwartung verstoßen
checks = {
    "Umsatz <= 0": (df["Total Revenue"] <= 0).sum(),
    "Kosten > Umsatz": (df["Cost of Revenue"] > df["Total Revenue"]).sum(),
    # Bruttogewinn sollte Umsatz minus Herstellkosten sein (Toleranz 1.000 USD für Rundung)
    "Bruttogewinn != Umsatz - Kosten": (
        (df["Total Revenue"] - df["Cost of Revenue"] - df["Gross Profit"]).abs() > 1e3
    ).sum(),
    # Banken/Versicherer haben keine Herstellkosten -> Bruttomarge dort nicht aussagekräftig
    "Cost of Revenue = 0": (df["Cost of Revenue"] == 0).sum(),
    "Gross Margin außerhalb 0-100 %": (~df["Gross Margin"].between(0, 100)).sum(),
}
pd.Series(checks, name="anzahl_zeilen").to_frame()

,anzahl_zeilen
Umsatz <= 0,0
Kosten > Umsatz,2
Bruttogewinn != Umsatz - Kosten,71
Cost of Revenue = 0,107
Gross Margin außerhalb 0-100 %,0


In [12]:
# Zeitliche Plausibilität: Der Datensatz soll etwa die Jahre 2012-2016 abdecken
period_year = pd.to_datetime(df["Period Ending"]).dt.year
print("Jahr des Periodenendes:")
print(period_year.value_counts().sort_index().to_string())

print("\nWerte in 'For Year' (NaN = fehlend):")
print(df["For Year"].value_counts(dropna=False).sort_index().to_string())

Jahr des Periodenendes:
Period Ending
2003      1
2004      1
2006      1
2007      1
2012    236
2013    439
2014    436
2015    445
2016    220
2017      1

Werte in 'For Year' (NaN = fehlend):
For Year
1,215.00      1
2,012.00    244
2,013.00    425
2,014.00    428
2,015.00    425
2,016.00     85
NaN         173


In [13]:
# Ausreißer genauer ansehen: Abschlüsse vor 2010 und das unmögliche Jahr 1215
df.loc[(period_year < 2010) | (df["For Year"] == 1215),
       ["Ticker Symbol", "Period Ending", "For Year", "Total Revenue"]]

,Ticker Symbol,Period Ending,For Year,Total Revenue
378,COTY,2003-06-30,NaN,"1,577,000.00"
379,COTY,2004-06-30,NaN,"1,514,000.00"
380,COTY,2006-02-28,NaN,"79,562,000.00"
381,COTY,2007-02-28,NaN,"99,642,000.00"
854,IPG,2015-12-31,"1,215.00","7,613,800,000.00"


## Fazit

- **Größe:** 1.781 Zeilen × 79 Spalten, 448 Unternehmen, meist 4 Geschäftsjahre je Unternehmen.
- **Spalten:** 76 numerische Kennzahlen, dazu Ticker und Periodenende (als Text). `Unnamed: 0` ist ein überflüssiger Index.
- **Fehlende Werte:** Liquiditätskennzahlen (Cash/Current/Quick Ratio) fehlen v. a. bei Finanzunternehmen, außerdem EPS und Aktienanzahl. Diese Lücken sind fachlich erklärbar und werden nicht geschätzt.
- **Plausibilität:** Die Größenordnungen stimmen (z. B. Apple). Aber:
  - `For Year` ist lückenhaft, enthält den Fehlwert 1215 und erzeugt Duplikate → das Geschäftsjahr wird in `clean.py` aus `Period Ending` abgeleitet.
  - COTY hat einzelne Abschlüsse aus 2003–2007 mit unplausiblen Umsätzen → werden entfernt.
  - Banken/Versicherer haben `Cost of Revenue = 0` → Kostenquote und Bruttomarge werden in `transform.py` als „nicht anwendbar“ behandelt.